# 06 — Clustering and Dimensionality Reduction

This notebook explores the **structure of build targets** by projecting their feature vectors into
low-dimensional spaces and applying clustering algorithms. The aim is to discover natural groupings
of targets that could inform merge/split decisions and identify outliers.

Pipeline:
1. Construct a feature matrix from target-level metrics and graph properties
2. PCA for variance decomposition
3. t-SNE / UMAP for 2-D visualisation
4. DBSCAN and hierarchical clustering
5. Cluster interpretation via centroid analysis
6. Outlier detection

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from build_optimiser.graph import load_graph, attach_metrics, critical_path, node_centrality
from build_optimiser.simulation import rebuild_cost, expected_daily_cost, simulate_merge, simulate_split, monte_carlo_rebuild_cost
from build_optimiser.config import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")

file_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "file_metrics.parquet")
target_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "target_metrics.parquet")
edge_list = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "edge_list.parquet")

G = load_graph(str(PROJECT_ROOT / "data" / "raw" / "dot"))
attach_metrics(G, target_metrics)

print(f"Files: {len(file_metrics)}, Targets: {len(target_metrics)}, Edges: {len(edge_list)}")

In [ ]:
# ── Feature matrix construction with StandardScaler ────────────────────

from sklearn.preprocessing import StandardScaler

# Enrich target_metrics with graph-derived features
centrality = node_centrality(G)
in_deg = dict(G.in_degree())
out_deg = dict(G.out_degree())
trans_deps = {t: len(nx.descendants(G, t)) for t in G.nodes}
trans_dependants = {t: len(nx.ancestors(G, t)) for t in G.nodes}

features = target_metrics.copy()
features["betweenness_centrality"] = features["cmake_target"].map(centrality).fillna(0)
features["in_degree"] = features["cmake_target"].map(in_deg).fillna(0)
features["out_degree"] = features["cmake_target"].map(out_deg).fillna(0)
features["transitive_deps"] = features["cmake_target"].map(trans_deps).fillna(0)
features["transitive_dependants"] = features["cmake_target"].map(trans_dependants).fillna(0)

# Select numeric columns for clustering
numeric_cols = features.select_dtypes(include=[np.number]).columns.tolist()
# Remove any ID-like columns
exclude_patterns = ["id", "index"]
numeric_cols = [c for c in numeric_cols if not any(p in c.lower() for p in exclude_patterns)]

X_raw = features[numeric_cols].fillna(0).values
target_names = features["cmake_target"].values

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f"Feature matrix: {X.shape[0]} targets x {X.shape[1]} features")
print(f"Features: {numeric_cols}")

In [ ]:
# ── PCA with explained variance ratio plot ─────────────────────────────

from sklearn.decomposition import PCA

n_components = min(X.shape[0], X.shape[1], 20)
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, n_components + 1), pca.explained_variance_ratio_, alpha=0.7, label="Individual")
axes[0].step(range(1, n_components + 1), np.cumsum(pca.explained_variance_ratio_),
             where="mid", color="red", lw=2, label="Cumulative")
axes[0].axhline(0.9, ls="--", color="grey", alpha=0.5)
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained Variance Ratio")
axes[0].set_title("PCA Scree Plot")
axes[0].legend()

# PC1 vs PC2 scatter
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], s=20, alpha=0.6, c="steelblue")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
axes[1].set_title("PCA – PC1 vs PC2")

plt.tight_layout()
plt.show()

n_90 = int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.9)) + 1
print(f"{n_90} components explain 90 % of variance")

# Feature loadings on first two PCs
loadings = pd.DataFrame(pca.components_[:2].T, index=numeric_cols, columns=["PC1", "PC2"])
print("\nTop loadings on PC1:")
display(loadings.reindex(loadings["PC1"].abs().sort_values(ascending=False).index).head(8))

In [ ]:
# ── t-SNE / UMAP 2D visualisation coloured by community ───────────────

from sklearn.manifold import TSNE

# Detect communities using Louvain on the undirected projection
G_undirected = G.to_undirected()
communities = nx.community.louvain_communities(G_undirected, seed=42)
community_map = {}
for i, comm in enumerate(communities):
    for node in comm:
        community_map[node] = i

community_labels = np.array([community_map.get(t, -1) for t in target_names])

# t-SNE
perplexity = min(30, max(5, X.shape[0] // 4))
tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# t-SNE coloured by community
n_communities = len(communities)
palette = sns.color_palette("husl", n_communities)
for ci in range(n_communities):
    mask = community_labels == ci
    axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1], s=25, alpha=0.7,
                    label=f"Community {ci}" if ci < 10 else None, color=palette[ci])
axes[0].set_title(f"t-SNE coloured by Louvain community ({n_communities} communities)")
axes[0].set_xlabel("t-SNE 1")
axes[0].set_ylabel("t-SNE 2")
if n_communities <= 10:
    axes[0].legend(fontsize=8, markerscale=2)

# Try UMAP if available, otherwise re-use t-SNE with different perplexity
try:
    import umap
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    X_umap = reducer.fit_transform(X)
    embed_label = "UMAP"
    X_2d = X_umap
except ImportError:
    # Fallback: use PCA 2-D
    X_2d = X_pca[:, :2]
    embed_label = "PCA (UMAP not installed)"

for ci in range(n_communities):
    mask = community_labels == ci
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], s=25, alpha=0.7, color=palette[ci])
axes[1].set_title(f"{embed_label} coloured by Louvain community")
axes[1].set_xlabel(f"{embed_label} 1")
axes[1].set_ylabel(f"{embed_label} 2")

plt.tight_layout()
plt.show()

In [ ]:
# ── DBSCAN clustering ──────────────────────────────────────────────────

from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

# Scan eps values to find a reasonable setting
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=min(5, X.shape[0]))
nn.fit(X)
distances, _ = nn.kneighbors(X)
k_distances = np.sort(distances[:, -1])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# k-distance plot for eps selection
axes[0].plot(k_distances, lw=2)
axes[0].set_xlabel("Points (sorted)")
axes[0].set_ylabel("k-distance")
axes[0].set_title("k-Distance Plot (elbow = good eps)")

# Choose eps at the elbow (use knee detection heuristic)
gradient = np.gradient(k_distances)
elbow_idx = np.argmax(gradient > np.median(gradient) * 2) if len(gradient) > 0 else len(k_distances) // 2
eps_value = k_distances[max(elbow_idx, 1)]
axes[0].axhline(eps_value, ls="--", color="red", alpha=0.6, label=f"eps = {eps_value:.2f}")
axes[0].legend()

dbscan = DBSCAN(eps=eps_value, min_samples=3)
db_labels = dbscan.fit_predict(X)
n_clusters = len(set(db_labels) - {-1})
n_noise = (db_labels == -1).sum()

axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=db_labels, cmap="tab20", s=25, alpha=0.7)
axes[1].set_title(f"DBSCAN on t-SNE space ({n_clusters} clusters, {n_noise} noise)")
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")

plt.tight_layout()
plt.show()

if n_clusters >= 2:
    non_noise = db_labels != -1
    sil = silhouette_score(X[non_noise], db_labels[non_noise])
    print(f"DBSCAN: {n_clusters} clusters, {n_noise} noise points, silhouette = {sil:.3f}")
else:
    print(f"DBSCAN: {n_clusters} clusters, {n_noise} noise points (not enough clusters for silhouette)")

In [ ]:
# ── Hierarchical clustering with dendrogram ────────────────────────────

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

# Ward linkage on scaled features
Z = linkage(X, method="ward")

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Full dendrogram (truncated to 30 leaves for readability)
dendrogram(Z, ax=axes[0], truncate_mode="lastp", p=30, leaf_rotation=90,
           leaf_font_size=8, show_contracted=True, color_threshold=0)
axes[0].set_title("Hierarchical Clustering Dendrogram (Ward, truncated)")
axes[0].set_ylabel("Distance")

# Cut at different k and plot silhouette
k_range = range(2, min(16, X.shape[0]))
silhouettes = []
for k in k_range:
    labels_k = fcluster(Z, t=k, criterion="maxclust")
    if len(set(labels_k)) >= 2:
        silhouettes.append(silhouette_score(X, labels_k))
    else:
        silhouettes.append(0)

axes[1].plot(list(k_range), silhouettes, "o-", lw=2, color="steelblue")
best_k = list(k_range)[np.argmax(silhouettes)]
axes[1].axvline(best_k, ls="--", color="red", alpha=0.6, label=f"Best k = {best_k}")
axes[1].set_xlabel("Number of clusters")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette Score vs Number of Clusters (Ward)")
axes[1].legend()

plt.tight_layout()
plt.show()

# Assign final labels
hier_labels = fcluster(Z, t=best_k, criterion="maxclust")
print(f"Optimal k = {best_k}, silhouette = {max(silhouettes):.3f}")
print(f"Cluster sizes: {dict(zip(*np.unique(hier_labels, return_counts=True)))}")

In [ ]:
# ── Cluster interpretation (centroid analysis) ─────────────────────────

features["hier_cluster"] = hier_labels

# Compute centroid statistics per cluster
centroid_df = features.groupby("hier_cluster")[numeric_cols].mean()

# Normalise centroids for heatmap visualisation
centroid_norm = (centroid_df - centroid_df.min()) / (centroid_df.max() - centroid_df.min() + 1e-12)

fig, ax = plt.subplots(figsize=(14, max(4, best_k * 0.8)))
sns.heatmap(centroid_norm.T, annot=False, cmap="YlOrRd", ax=ax,
            xticklabels=[f"Cluster {c}" for c in centroid_norm.index],
            yticklabels=numeric_cols, linewidths=0.5)
ax.set_title("Cluster Centroids (normalised feature values)")
plt.tight_layout()
plt.show()

# Annotate each cluster with a summary
print("Cluster profiles:")
for cluster_id in sorted(features["hier_cluster"].unique()):
    members = features[features["hier_cluster"] == cluster_id]
    top_features = centroid_norm.loc[cluster_id].nlargest(3)
    desc = ", ".join([f"{f} (high)" for f in top_features.index])
    print(f"  Cluster {cluster_id} ({len(members)} targets): characterised by {desc}")
    print(f"    Examples: {', '.join(members['cmake_target'].head(5).tolist())}")

In [ ]:
# ── Outlier detection ──────────────────────────────────────────────────

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

# Method 1: Isolation Forest
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso_labels = iso.fit_predict(X)
iso_scores = iso.decision_function(X)

# Method 2: Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=min(20, X.shape[0] - 1), contamination=0.05)
lof_labels = lof.fit_predict(X)
lof_scores = -lof.negative_outlier_factor_  # higher = more outlier-ish

# Combine: flag targets that both methods consider outliers
features["iso_outlier"] = iso_labels == -1
features["lof_outlier"] = lof_labels == -1
features["iso_score"] = iso_scores
features["lof_score"] = lof_scores
features["consensus_outlier"] = features["iso_outlier"] & features["lof_outlier"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Isolation Forest outliers on t-SNE
colors_iso = np.where(iso_labels == -1, "red", "steelblue")
axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=colors_iso, s=25, alpha=0.7)
axes[0].set_title(f"Isolation Forest outliers ({(iso_labels == -1).sum()} flagged)")
axes[0].set_xlabel("t-SNE 1")
axes[0].set_ylabel("t-SNE 2")

# LOF outliers on t-SNE
colors_lof = np.where(lof_labels == -1, "red", "steelblue")
axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=colors_lof, s=25, alpha=0.7)
axes[1].set_title(f"Local Outlier Factor outliers ({(lof_labels == -1).sum()} flagged)")
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")

plt.tight_layout()
plt.show()

outliers = features[features["consensus_outlier"]][["cmake_target", "iso_score", "lof_score"] + numeric_cols[:5]]
print(f"\nConsensus outliers ({len(outliers)} targets):")
display(outliers.sort_values("iso_score"))